In [1]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from scapy.utils import PcapReader
from scapy.layers.inet import IP, TCP, UDP, ICMP
from scapy.layers.l2 import ARP

In [2]:
DATASET_PATH = Path("/home/emanuele/Documentos/ProjetosFaculdade/ProjetoIoT/idle-2021/traffic-idle-2021")

pcaps = [
    p for p in DATASET_PATH.rglob("*.pcap")
    if not p.name.startswith("._")
]

print(f"Arquivos encontrados: {len(pcaps)}")

Arquivos encontrados: 297


In [3]:
def extrair_features(caminho):
    total_packets = 0
    total_bytes = 0

    tcp = udp = icmp = arp = dns = 0
    tamanhos = []

    ips_origem = set()
    ips_destino = set()
    portas = set()

    primeiro_tempo = None
    ultimo_tempo = None

    with PcapReader(str(caminho)) as captura:
        for p in captura:
            total_packets += 1

            tamanho = len(p)
            total_bytes += tamanho
            tamanhos.append(tamanho)

            tempo = float(p.time)
            if primeiro_tempo is None:
                primeiro_tempo = tempo
            ultimo_tempo = tempo

            if IP in p:
                ips_origem.add(p[IP].src)
                ips_destino.add(p[IP].dst)

            if TCP in p:
                tcp += 1
                portas.add(p[TCP].dport)

            if UDP in p:
                udp += 1
                portas.add(p[UDP].dport)

                if p[UDP].sport == 53 or p[UDP].dport == 53:
                    dns += 1

            if ICMP in p:
                icmp += 1

            if ARP in p:
                arp += 1

    duracao = 0
    if primeiro_tempo is not None and ultimo_tempo is not None:
        duracao = ultimo_tempo - primeiro_tempo

    return {
        "device": caminho.parents[1].name,
        "arquivo": caminho.name,
        "total_packets": total_packets,
        "total_bytes": total_bytes,
        "avg_packet_size": total_bytes / total_packets if total_packets else 0,
        "max_packet_size": max(tamanhos) if tamanhos else 0,
        "min_packet_size": min(tamanhos) if tamanhos else 0,
        "tcp_packets": tcp,
        "udp_packets": udp,
        "dns_packets": dns,
        "icmp_packets": icmp,
        "arp_packets": arp,
        "unique_source_ips": len(ips_origem),
        "unique_destination_ips": len(ips_destino),
        "unique_ports": len(portas),
        "capture_duration": duracao,
        "packets_per_second": total_packets / duracao if duracao > 0 else 0,
        "bytes_per_second": total_bytes / duracao if duracao > 0 else 0,
        "tcp_ratio": tcp / total_packets if total_packets else 0,
        "udp_ratio": udp / total_packets if total_packets else 0,
        "dns_ratio": dns / total_packets if total_packets else 0,
        "icmp_ratio": icmp / total_packets if total_packets else 0,
    }

In [5]:
teste = extrair_features(pcaps[0])
teste

{'device': 'google-home-mini',
 'arquivo': '2021-09-02_15.50.20_192.168.12.128.pcap',
 'total_packets': 33474,
 'total_bytes': 3115439,
 'avg_packet_size': 93.07041285774034,
 'max_packet_size': 1484,
 'min_packet_size': 62,
 'tcp_packets': 2568,
 'udp_packets': 29590,
 'dns_packets': 834,
 'icmp_packets': 1316,
 'arp_packets': 0,
 'unique_source_ips': 29,
 'unique_destination_ips': 31,
 'unique_ports': 328,
 'capture_duration': 10218.833552837372,
 'packets_per_second': 3.2757163356189096,
 'bytes_per_second': 304.87227176089624,
 'tcp_ratio': 0.07671625739379817,
 'udp_ratio': 0.8839696480850809,
 'dns_ratio': 0.024914859293780248,
 'icmp_ratio': 0.03931409452112087}

In [6]:
import time

inicio = time.time()
teste = extrair_features(pcaps[0])
fim = time.time()

print(teste)
print(f"Tempo: {fim - inicio:.2f} segundos")

{'device': 'google-home-mini', 'arquivo': '2021-09-02_15.50.20_192.168.12.128.pcap', 'total_packets': 33474, 'total_bytes': 3115439, 'avg_packet_size': 93.07041285774034, 'max_packet_size': 1484, 'min_packet_size': 62, 'tcp_packets': 2568, 'udp_packets': 29590, 'dns_packets': 834, 'icmp_packets': 1316, 'arp_packets': 0, 'unique_source_ips': 29, 'unique_destination_ips': 31, 'unique_ports': 328, 'capture_duration': 10218.833552837372, 'packets_per_second': 3.2757163356189096, 'bytes_per_second': 304.87227176089624, 'tcp_ratio': 0.07671625739379817, 'udp_ratio': 0.8839696480850809, 'dns_ratio': 0.024914859293780248, 'icmp_ratio': 0.03931409452112087}
Tempo: 5.20 segundos


In [7]:
from tqdm import tqdm

dataset = []

for pcap in tqdm(pcaps):
    try:
        dataset.append(extrair_features(pcap))
    except Exception as e:
        print(f"Erro em {pcap}: {e}")

df = pd.DataFrame(dataset)

df.head()

100%|██████████| 297/297 [30:33<00:00,  6.17s/it] 


,device,arquivo,total_packets,total_bytes,avg_packet_size,max_packet_size,min_packet_size,tcp_packets,udp_packets,dns_packets,...,unique_source_ips,unique_destination_ips,unique_ports,capture_duration,packets_per_second,bytes_per_second,tcp_ratio,udp_ratio,dns_ratio,icmp_ratio
0,google-home-mini,2021-09-02_15.50.20_192.168.12.128.pcap,33474,3115439,93.070413,1484,62,2568,29590,834,...,29,31,328,10218.833553,3.275716,304.872272,0.076716,0.883970,0.024915,0.039314
1,google-home-mini,2021-09-06_15.50.20_192.168.12.128.pcap,209059,22187779,106.131661,1484,54,54002,143960,7024,...,43,46,2592,86400.516961,2.419650,256.801461,0.258310,0.688609,0.033598,0.053038
2,google-home-mini,2021-09-07_15.50.20_192.168.12.128.pcap,52013,11839414,227.624132,1484,54,24396,17841,6214,...,45,49,2334,76169.694660,0.682857,155.434705,0.469037,0.343010,0.119470,0.187818
3,google-home-mini,2021-09-04_15.50.20_192.168.12.128.pcap,489973,41024647,83.728383,1514,54,147059,331820,7030,...,41,45,2586,86400.032354,5.670982,474.822125,0.300137,0.677221,0.014348,0.022626
4,google-home-mini,2021-09-05_15.50.20_192.168.12.128.pcap,352071,31586693,89.716827,1514,54,66000,275011,6988,...,43,47,2584,86399.138360,4.074936,365.590370,0.187462,0.781124,0.019848,0.031391


In [8]:
df.to_csv("csv/dataset_iot.csv", index=False)

print("CSV salvo com sucesso!")
print(df.shape)

CSV salvo com sucesso!
(297, 22)


In [9]:
df.shape

(297, 22)

In [10]:
df.head()

,device,arquivo,total_packets,total_bytes,avg_packet_size,max_packet_size,min_packet_size,tcp_packets,udp_packets,dns_packets,...,unique_source_ips,unique_destination_ips,unique_ports,capture_duration,packets_per_second,bytes_per_second,tcp_ratio,udp_ratio,dns_ratio,icmp_ratio
0,google-home-mini,2021-09-02_15.50.20_192.168.12.128.pcap,33474,3115439,93.070413,1484,62,2568,29590,834,...,29,31,328,10218.833553,3.275716,304.872272,0.076716,0.883970,0.024915,0.039314
1,google-home-mini,2021-09-06_15.50.20_192.168.12.128.pcap,209059,22187779,106.131661,1484,54,54002,143960,7024,...,43,46,2592,86400.516961,2.419650,256.801461,0.258310,0.688609,0.033598,0.053038
2,google-home-mini,2021-09-07_15.50.20_192.168.12.128.pcap,52013,11839414,227.624132,1484,54,24396,17841,6214,...,45,49,2334,76169.694660,0.682857,155.434705,0.469037,0.343010,0.119470,0.187818
3,google-home-mini,2021-09-04_15.50.20_192.168.12.128.pcap,489973,41024647,83.728383,1514,54,147059,331820,7030,...,41,45,2586,86400.032354,5.670982,474.822125,0.300137,0.677221,0.014348,0.022626
4,google-home-mini,2021-09-05_15.50.20_192.168.12.128.pcap,352071,31586693,89.716827,1514,54,66000,275011,6988,...,43,47,2584,86399.138360,4.074936,365.590370,0.187462,0.781124,0.019848,0.031391


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   device                  297 non-null    object 
 1   arquivo                 297 non-null    object 
 2   total_packets           297 non-null    int64  
 3   total_bytes             297 non-null    int64  
 4   avg_packet_size         297 non-null    float64
 5   max_packet_size         297 non-null    int64  
 6   min_packet_size         297 non-null    int64  
 7   tcp_packets             297 non-null    int64  
 8   udp_packets             297 non-null    int64  
 9   dns_packets             297 non-null    int64  
 10  icmp_packets            297 non-null    int64  
 11  arp_packets             297 non-null    int64  
 12  unique_source_ips       297 non-null    int64  
 13  unique_destination_ips  297 non-null    int64  
 14  unique_ports            297 non-null    in